# 02 - Bronze ingest

Incrementally loads new scraper CSVs from the landing volume into `bronze.jobs_raw` with **Auto Loader** (`availableNow`).

* Auto Loader keeps processed files in its checkpoint, so every file is ingested exactly once, even if the job is triggered several times.
* All business columns are kept as `STRING` exactly as in the file; typing and cleaning happen in silver.
* Values that don't fit the schema go to `_rescued_data` instead of failing the run.
* Debug columns: `_source_file`, `_source_system`, `_landing_folder_date`, `_file_modification_time`, `_file_size`, `_row_hash`, `_ingested_at`, `_job_run_id`.

**Job task environment:** no extra dependencies needed.

In [ ]:
%run ./_common

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("landing_path", "/Volumes/jobseeker/default/scrapes", "Landing volume path")
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"], "Full refresh (truncate bronze + reset checkpoint)")

LANDING_PATH = dbutils.widgets.get("landing_path").rstrip("/")
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"

if not LANDING_PATH.startswith("/Volumes/"):
    raise ValueError(f"landing_path must be a Unity Catalog volume path: {LANDING_PATH}")

BRONZE_TABLE = table("bronze", "jobs_raw")
CHECKPOINT = f"{PIPELINE_VOLUME}/checkpoints/bronze_jobs_raw"

BUSINESS_COLUMNS = [
    "company_name", "ats_type", "company_url", "job_id", "title", "location",
    "posted_date", "job_url", "description", "collected_on", "posted_days",
]
CSV_SCHEMA = ", ".join(f"{column} STRING" for column in BUSINESS_COLUMNS)

print(f"landing={LANDING_PATH}\ncheckpoint={CHECKPOINT}\ntarget={BRONZE_TABLE}\nfull_refresh={FULL_REFRESH}")

## Optional full refresh
Only for development: empties bronze and resets the checkpoint so every file in the volume is ingested again. Silver dedupes by `job_key`, so re-ingesting is safe but slower.

In [ ]:
if FULL_REFRESH:
    spark.sql(f"TRUNCATE TABLE {BRONZE_TABLE}")
    dbutils.fs.rm(CHECKPOINT, recurse=True)
    print("Bronze truncated and checkpoint removed")

## Stream new files into bronze

In [ ]:
with task_run("bronze_ingest") as metrics:
    raw = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .option("pathGlobFilter", "*.csv")
        .option("rescuedDataColumn", "_rescued_data")
        .schema(CSV_SCHEMA)
        .load(LANDING_PATH)
    )

    row_hash = F.sha2(F.concat_ws("||", *[F.coalesce(F.col(column), F.lit("")) for column in BUSINESS_COLUMNS]), 256)
    landing_folder_date = F.expr(
        "CAST(try_to_timestamp(regexp_extract(_metadata.file_path, '/([0-9]{8})/[^/]+$', 1), 'ddMMyyyy') AS DATE)"
    )

    bronze = raw.select(
        *BUSINESS_COLUMNS,
        F.col("_rescued_data"),
        F.col("_metadata.file_path").alias("_source_file"),
        F.lower(F.split(F.col("_metadata.file_name"), "_").getItem(0)).alias("_source_system"),
        landing_folder_date.alias("_landing_folder_date"),
        F.col("_metadata.file_modification_time").alias("_file_modification_time"),
        F.col("_metadata.file_size").alias("_file_size"),
        row_hash.alias("_row_hash"),
        F.current_timestamp().alias("_ingested_at"),
        F.lit(JOB_RUN_ID).alias("_job_run_id"),
    )

    query = (
        bronze.writeStream
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
    )
    query.awaitTermination()

    this_run = spark.table(BRONZE_TABLE).where(F.col("_job_run_id") == JOB_RUN_ID)
    stats = this_run.agg(
        F.count("*").alias("rows"),
        F.countDistinct("_source_file").alias("files"),
        F.sum(F.when(F.col("_rescued_data").isNotNull(), 1).otherwise(0)).alias("rescued_rows"),
    ).first()
    by_source = {row["_source_system"]: row["count"] for row in this_run.groupBy("_source_system").count().collect()}

    metrics.update(
        files_read=int(stats["files"]),
        rows_in=int(stats["rows"]),
        rows_out=int(stats["rows"]),
        rows_rejected=int(stats["rescued_rows"] or 0),
        details={"by_source": by_source, "rescued_rows": int(stats["rescued_rows"] or 0)},
        message="no new files" if stats["rows"] == 0 else None,
    )

In [ ]:
stop_if_job()  # scheduled runs end here; the cells below are for manual inspection

## Inspect this run

In [ ]:
display(
    spark.table(BRONZE_TABLE)
    .where(F.col("_job_run_id") == JOB_RUN_ID)
    .groupBy("_source_system", "_landing_folder_date", "_source_file")
    .agg(F.count("*").alias("rows"), F.sum(F.when(F.col("_rescued_data").isNotNull(), 1).otherwise(0)).alias("rescued_rows"))
    .orderBy("_landing_folder_date", "_source_system", "_source_file")
)

In [ ]:
display(spark.table(table("ops", "pipeline_runs")).orderBy(F.col("started_at").desc()).limit(10))